<a href="https://colab.research.google.com/github/siddharthsinh-dev/iu-bsc-thesis-llm-comparison/blob/main/notebooks/exp2_dataset_preparation_fnspid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Thesis: A Comparative Study of Large Language Models for Financial Sentiment Analysis and Their Predictive Potential for Short-Term Stock Price Movement

Component: Experiment 2 - Dataset Preparation

Description: This notebook filters and samples the FNSPID dataset (Dong et al., 2024) for 10 selected companies across the period January 2018 to June 2020 (last update to the dataset). It uses FinBERT to remove neutral headlines, randomly samples 100 headlines per company (1000 total headlines), retrieves next-day forward stock price movement from Yahoo Finance, merges both datasets and saves the final dataset as a CSV file to evaluate the four LLM models on it.

Select T4 GPU as runtime.

In [ ]:
# Install all required libraries — Dataset Preparation

!pip install -q transformers datasets torch
!pip install -q yfinance pandas matplotlib seaborn

print("All libraries installed successfully.")

In [ ]:
# Import required libraries for dataset preparation

import pandas as pd
import numpy as np
import torch
import warnings

import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline

warnings.filterwarnings("ignore")

# Check GPU
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected    : {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected")

Add the API Key from Hugging Face using "Add New Secret" in Google Colab

In [ ]:
# Connect to Hugging Face
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

print("Hugging Face login successful.")

In [ ]:
# Mount Google Drive and download FNSPID dataset
# Source: Dong et al. (2024)

from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import hf_hub_download
import os

print("Downloading FNSPID dataset...")

file_path_all = hf_hub_download(
    repo_id="Zihan1004/FNSPID",
    filename="Stock_news/All_external.csv",
    repo_type="dataset",
    token=hf_token
)

size_gb = os.path.getsize(file_path_all) / (1024**3)
print(f"Downloaded successfully.")
print(f"File size: {size_gb:.2f} GB")

In [ ]:
# Filter FNSPID for 10 tickers and date range 2018-2020

TICKERS_FINAL = ["AAPL", "AMZN", "GOOGL", "INTC", "JPM",
                 "NFLX", "NVDA", "TSLA", "ORCL", "FB"]

chunks_final = []
total_read = 0

print("Filtering FNSPID for 10 tickers and 2018-2020...")

for chunk in pd.read_csv(
    file_path_all,
    chunksize=50000,
    encoding="latin-1",
    on_bad_lines="skip",
    engine="python"
):
    filtered = chunk[chunk["Stock_symbol"].isin(TICKERS_FINAL)].copy()

    if len(filtered) > 0:
        filtered["Date"] = pd.to_datetime(
            filtered["Date"], errors="coerce", utc=True
        ).dt.tz_localize(None)

        filtered = filtered[
            (filtered["Date"] >= "2018-01-01") &
            (filtered["Date"] <= "2020-06-30")
        ]

        if len(filtered) > 0:
            chunks_final.append(
                filtered[["Stock_symbol", "Date", "Article_title"]]
            )

    total_read += len(chunk)
    if total_read % 2000000 == 0:
        print(f"  Processed {total_read:,} rows...")

df_fnspid_filtered = pd.concat(chunks_final).reset_index(drop=True)

print(f"\nDone.")
print(f"Total rows : {len(df_fnspid_filtered):,}")
print(f"\nPer ticker counts:")
print(df_fnspid_filtered["Stock_symbol"].value_counts())

In [ ]:
# Load FinBERT and classify all headlines
# Keep only positive and negative headlines - exclude neutral headlines

print("Loading FinBERT for pre-filtering...")

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0 if torch.cuda.is_available() else -1
)

print("FinBERT loaded. Running on all headlines...")

headlines = df_fnspid_filtered["Article_title"].tolist()

finbert_results = finbert(headlines, batch_size=32)
df_fnspid_filtered["finbert_sentiment"] = [r["label"].lower() for r in finbert_results]

# Keep only positive and negative
df_fnspid_sentiment = df_fnspid_filtered[
    df_fnspid_filtered["finbert_sentiment"] != "neutral"
].copy()

print(f"\nTotal headlines before filter : {len(df_fnspid_filtered):,}")
print(f"After removing neutral        : {len(df_fnspid_sentiment):,}")
print(f"Neutral removed               : {len(df_fnspid_filtered) - len(df_fnspid_sentiment):,}")
print(f"\nSentiment distribution:")
print(df_fnspid_sentiment["finbert_sentiment"].value_counts())
print(f"\nPer ticker after filter:")
print(df_fnspid_sentiment["Stock_symbol"].value_counts())

In [ ]:
# Download stock price data and compute next-day movement labels

print("Downloading stock price data...")

tickers_without_fb = ["AAPL", "AMZN", "GOOGL", "INTC", "JPM",
                       "NFLX", "NVDA", "ORCL", "TSLA"]

price_main = yf.download(
    tickers_without_fb,
    start="2018-01-01",
    end="2020-07-01",
    progress=False
)["Close"]

# FB/Meta separately
price_fb = yf.download(
    "META",
    start="2018-01-01",
    end="2020-07-01",
    progress=False
)["Close"]

price_fb.name = "FB"

# Combine
price_data = price_main.copy()
price_data["FB"] = price_fb

# Reshape to long format
price_long = price_data.reset_index().melt(
    id_vars="Date",
    var_name="ticker",
    value_name="close_price"
)

price_long["date"] = pd.to_datetime(price_long["Date"]).dt.date
price_long = price_long.drop(columns="Date")
price_long = price_long.sort_values(["ticker", "date"]).reset_index(drop=True)

# Next-day movement label (1 trading day forward)
price_long["next_close_1d"] = price_long.groupby("ticker")["close_price"].shift(-1)
price_long["movement"] = (price_long["next_close_1d"] > price_long["close_price"]).astype(int)
price_long = price_long.dropna(subset=["next_close_1d"])

print(f"Price data downloaded.")
print(f"Total rows     : {len(price_long):,}")
print(f"Tickers        : {price_long['ticker'].nunique()}")
print(f"\nNext-day movement distribution:")
print(price_long["movement"].value_counts())

In [ ]:
# Sample 100 sentiment-bearing headlines per company
# From trading days only, then merge with next-day price movement

trading_days = set(price_long["date"].unique())

# Filter to trading days only
df_fnspid_trading = df_fnspid_sentiment[
    df_fnspid_sentiment["Date"].dt.date.isin(trading_days)
].copy()

print("Sentiment-bearing headlines on trading days per ticker:")
print(df_fnspid_trading["Stock_symbol"].value_counts())
print(f"\nTotal available: {len(df_fnspid_trading):,}")

# Sample 100 per company
SAMPLES_PER_COMPANY = 100
sampled_parts = []

for ticker in TICKERS_FINAL:
    df_ticker = df_fnspid_trading[
        df_fnspid_trading["Stock_symbol"] == ticker
    ]
    n = min(SAMPLES_PER_COMPANY, len(df_ticker))
    sampled = df_ticker.sample(n=n, random_state=42)
    sampled_parts.append(sampled)

df_sample = pd.concat(sampled_parts).reset_index(drop=True)
df_sample = df_sample[["Stock_symbol", "Date", "Article_title", "finbert_sentiment"]].copy()
df_sample.columns = ["ticker", "date", "headline", "finbert_sentiment"]
df_sample["date"] = pd.to_datetime(df_sample["date"]).dt.date

# Merge with price data
df_merged = df_sample.merge(
    price_long[["ticker", "date", "close_price", "next_close_1d", "movement"]],
    on=["ticker", "date"],
    how="inner"
)

print("\n" + "=" * 45)
print("Final Dataset Statistics")
print("=" * 45)
print(f"  Headlines sampled   : {len(df_sample):,}")
print(f"  Matched with prices : {len(df_merged):,}")
print(f"  Unmatched           : {len(df_sample) - len(df_merged):,}")
print(f"\nPer ticker:")
print(df_merged["ticker"].value_counts().sort_index())
print(f"\nNext-day movement distribution:")
print(df_merged["movement"].value_counts())
print(f"\nSentiment distribution:")
print(df_merged["finbert_sentiment"].value_counts())

In [ ]:
# Save final dataset to Google Drive for evaluating four LLM models

import os
os.makedirs("/content/drive/MyDrive/Thesis_Data", exist_ok=True)

df_merged.to_csv("/content/drive/MyDrive/Thesis_Data/exp2_dataset.csv", index=False)

print("Saved to Google Drive — exp2_dataset.csv")
print(f"\nColumns: {list(df_merged.columns)}")
print(f"\nSample rows:")
print(df_merged.head(3))